Parse the outputs into final catalog files

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import time, sys, os
import numpy as np
import numpy.ma as ma

from astropy.table import Table, Column, MaskedColumn
from astropy.io import fits
from astropy.nddata import Cutout2D
from astropy.wcs import WCS
from astropy import units as u
from astropy.coordinates import SkyCoord

from collections import OrderedDict
from datetime import date

In [ ]:
## if no lensing, then remove all relevant blocks
save_mu_from_maps = True # save lensing mu from map
# save_mu_from_maps = False # save analytic mu

In [ ]:
ver = 'v3.0.2_LW_SUPER'
sps_ver = 'spsv2.0'

prior = 'phisfh'
# prior = 'phisfhzfixed'

cat = Table.read('../phot_catalog/UNCOVER_{}_CATALOG.fits'.format(ver))

fver = ver+'_'+sps_ver
print('fver', fver)

idx_usephot = cat['use_phot'] == 1

In [ ]:
if 'zfixed' in prior:
    flens = np.loadtxt('../sps_catalog/UNCOVER-SPS-catalog_v3.0.2_zspec_magnifications_shear.txt')
else:
    flens = np.loadtxt('../sps_catalog/UNCOVER-SPS-catalog_v3.0.2_zphot_magnifications_shear.txt')

In [ ]:
lens_cols = ['ID', 'mu', 'mu_low_68', 'mu_high_68', 'mu_low_95', 'mu_high_95', 
             'mu_rad', 'mu_rad_low_68', 'mu_rad_high_68', 'mu_rad_low_95', 'mu_rad_high_95',
             'mu_t', 'mu_t_low_68', 'mu_t_high_68', 'mu_t_low_95', 'mu_t_high_95', 
             'gamma1', 'gamma1_low_68', 'gamma1_high_68', 'gamma1_low_95', 'gamma1_high_95', 
             'gamma2', 'gamma2_low_68', 'gamma2_high_68', 'gamma2_low_95', 'gamma2_high_95',
             'theta', 'theta_low_68', 'theta_high_68', 'theta_low_95', 'theta_high_95']

In [ ]:
dlens = {}
for i, key in enumerate(lens_cols):
    dlens[key] = flens[:,i]
    

In [ ]:
fperc = np.load('results/quant_{}_post_parrot_{}.npz'.format(prior, fver), allow_pickle=True)
fspec = np.load('results/spec_{}_post_parrot_{}.npz'.format(prior, fver), allow_pickle=True)
fperc_gz = np.load('results/quant_gz_{}_post_parrot_{}.npz'.format(prior, fver), allow_pickle=True)

argsorted = np.argsort(fperc['objid'])
id_sort = fperc['objid'][argsorted]
dperc = {}
for key in fperc.files:
    dperc[key] = fperc[key][argsorted]
    
argsorted = np.argsort(fperc_gz['objid'])
id_sort = fperc_gz['objid'][argsorted]
dperc_gz = {}
for key in fperc_gz.files:
    dperc_gz[key] = fperc_gz[key][argsorted]


In [ ]:
argsorted = np.argsort(fspec['objid'])
id_sort = fspec['objid'][argsorted]
fspec.files.pop(-3) # rm eline_wave
dspec = {}
for key in fspec.files:
    print(key)
#     if key not in ['eline_flux_map', 'eline_flux_med', 'chi2_parrot', 'parrot_mags']:
    if key not in ['chi2_parrot', 'parrot_mags']:
        dspec[key] = fspec[key][argsorted]

In [ ]:
cat[0]

In [ ]:
keep_colnames = ['id', 'ra', 'dec', 
                 'id_DR1', #'match_radius_DR1',
                 'id_INT_v2', #'match_radius_INT_v2', 
                 'id_msa', 'id_alma', 
                 'use_aper', 'use_phot', 
                 'flag_kron', 
                 #'flag_nophot', 'flag_lowsnr', 'flag_star', 'flag_artifact', 'flag_nearbcg'
                ]
rm_cols = []
for i in range(len(cat.colnames)):
    if cat.colnames[i] not in keep_colnames:
        rm_cols.append(cat.colnames[i])

In [ ]:
cat.remove_columns(rm_cols)

In [ ]:
# Returns the current local date
today = date.today()
print("Today date is: ", today)

meta = OrderedDict()
meta['AUTHOR'] = 'Bingjie Wang; bwang@psu.edu'
meta['CREATED'] = str(today)

cat.meta = meta

In [ ]:
idx_finished = dperc['objid'] - 1 # only works if running on full catalog

# in first but not in second
id_not_finished = list(set(cat['id']) - set(dperc['objid']))
idx_not_finished = np.array(id_not_finished) - 1

# len(cat[cat['use_phot']==1]), np.sum(cat[idx_finished]['use_phot']), np.sum(cat[idx_not_finished]['use_phot'])

In [ ]:
idx_not_finished

In [ ]:
# np.sum(idx_finished == idx_finished[np.argsort(idx_finished)]), len(idx_finished)

In [ ]:
def fill_col(data, idx_finished=idx_finished, idx_not_finished=idx_not_finished):
    new_arr = np.ones(len(cat['id']))
    if len(idx_not_finished) > 0:
        new_arr[idx_not_finished] = np.nan
    
    new_arr[idx_finished] = np.copy(data)
    mask = np.ones_like(new_arr, dtype=bool)
    mask[idx_finished] = 0
    return new_arr, mask


In [ ]:
idx_finished_spec = dspec['objid'] - 1

# double checking
id_not_finished_spec = list(set(cat['id']) - set(dspec['objid']))
idx_not_finished_spec = np.array(id_not_finished_spec) - 1

len(cat[cat['use_phot']==1]), np.sum(cat[idx_finished_spec]['use_phot']), np.sum(cat[idx_not_finished_spec]['use_phot'])

In [ ]:
ii16 = len(dperc['zred'][0])//2 - 1
ii50 = len(dperc['zred'][0])//2
ii84 = len(dperc['zred'][0])//2 + 1

In [ ]:
_data, _mask = fill_col(dperc['zred_spec'])
# col_a = Column(name='z_spec', data=_data)
col_a = MaskedColumn(data=_data, name='z_spec', mask=_mask)
cat.add_columns([col_a])

_data, _mask = fill_col(dperc['zred_ml'], idx_finished=idx_finished_spec, idx_not_finished=idx_not_finished_spec)
col_a = MaskedColumn(name='z_ml', data=_data, mask=_mask)
cat.add_columns([col_a])

In [ ]:
thetas = ['zred', 'total_mass', 'stellar_mass', 
          'met', 'mwa', 'dust2', 'dust_index', 'dust1_fraction', 
          'log_fagn', 
          'sfr10', 'sfr30', 'sfr100', 
          'ssfr10', 'ssfr30', 'ssfr100', 
         ]
if save_mu_from_maps:
    thetas += ['mu']

theta_colnames = ['z_16', 'z_50', 'z_84', 
                  'mtot_16', 'mtot_50', 'mtot_84', 
                  'mstar_16', 'mstar_50', 'mstar_84', 
                  'met_16', 'met_50', 'met_84',
                  'mwa_16', 'mwa_50', 'mwa_84', 
                  'dust2_16', 'dust2_50', 'dust2_84',
                  'dust_index_16', 'dust_index_50', 'dust_index_84', 
                  'dust1_fraction_16', 'dust1_fraction_50', 'dust1_fraction_84',
                  'logfagn_16', 'logfagn_50', 'logfagn_84',
                  'sfr10_16', 'sfr10_50', 'sfr10_84',
                  'sfr30_16', 'sfr30_50', 'sfr30_84',
                  'sfr100_16', 'sfr100_50', 'sfr100_84',
                  'ssfr10_16', 'ssfr10_50', 'ssfr10_84',
                  'ssfr30_16', 'ssfr30_50', 'ssfr30_84',
                  'ssfr100_16', 'ssfr100_50', 'ssfr100_84'
                 ]
if save_mu_from_maps:
    theta_colnames += ['mu_num_16', 'mu_num_50', 'mu_num_84']
    
theta_col_units = [None, None, None, 
                   'log Msol', 'log Msol', 'log Msol', 
                   'log Msol', 'log Msol', 'log Msol', 
                   'log Zsol', 'log Zsol', 'log Zsol',
                   u.Gyr, u.Gyr, u.Gyr,
                   None, None, None, None, None, None, None, None, None, None, None, None, 
                   u.solMass/u.yr, u.solMass/u.yr, u.solMass/u.yr, 
                   u.solMass/u.yr, u.solMass/u.yr, u.solMass/u.yr, 
                   u.solMass/u.yr, u.solMass/u.yr, u.solMass/u.yr, 
                   1/u.yr, 1/u.yr, 1/u.yr, 
                   1/u.yr, 1/u.yr, 1/u.yr, 
                   1/u.yr, 1/u.yr, 1/u.yr, 
                  ]
if save_mu_from_maps:
    theta_col_units += [None, None, None]

dict_thetas = {}
dict_thetas['zred'] = ['z_16', 'z_50', 'z_84']
dict_thetas['total_mass'] = ['mtot_16', 'mtot_50', 'mtot_84']
dict_thetas['stellar_mass'] = ['mstar_16', 'mstar_50', 'mstar_84']
dict_thetas['met'] = ['met_16', 'met_50', 'met_84']
dict_thetas['mwa'] = ['mwa_16', 'mwa_50', 'mwa_84']
dict_thetas['dust2'] = ['dust2_16', 'dust2_50', 'dust2_84']
dict_thetas['dust_index'] = ['dust_index_16', 'dust_index_50', 'dust_index_84']
dict_thetas['dust1_fraction'] = ['dust1_fraction_16', 'dust1_fraction_50', 'dust1_fraction_84']
dict_thetas['log_fagn'] = ['logfagn_16', 'logfagn_50', 'logfagn_84']
dict_thetas['sfr10'] = ['sfr10_16', 'sfr10_50', 'sfr10_84']
dict_thetas['sfr30'] = ['sfr30_16', 'sfr30_50', 'sfr30_84']
dict_thetas['sfr100'] = ['sfr100_16', 'sfr100_50', 'sfr100_84']
dict_thetas['ssfr10'] = ['ssfr10_16', 'ssfr10_50', 'ssfr10_84']
dict_thetas['ssfr30'] = ['ssfr30_16', 'ssfr30_50', 'ssfr30_84']
dict_thetas['ssfr100'] = ['ssfr100_16', 'ssfr100_50', 'ssfr100_84']
if save_mu_from_maps:
    dict_thetas['mu'] = ['mu_num_16', 'mu_num_50', 'mu_num_84']

kk = 0
for t in thetas:
    for i_dict, ii in enumerate(np.array([ii16, ii50, ii84])):
        _data, _mask = fill_col(dperc[t][:,ii])
        col_a = MaskedColumn(name=dict_thetas[t][i_dict], data=_data, mask=_mask, unit=theta_col_units[kk])
        cat.add_columns([col_a])
        
        kk += 1

In [ ]:
_data, _mask = fill_col(dspec['mu_map'], idx_finished=idx_finished_spec, idx_not_finished=idx_not_finished_spec)
col_a = MaskedColumn(name='mu_ml', data=_data, mask=_mask)
cat.add_columns([col_a])

In [ ]:
thetas = ['rest_U', 'rest_V', 'rest_J', 'rest_u', 'rest_g', 'rest_i']

theta_colnames = ['rest_U_16', 'rest_U_50', 'rest_U_84',
                  'rest_V_16', 'rest_V_50', 'rest_V_84',
                  'rest_J_16', 'rest_J_50', 'rest_J_84',
                  'rest_u_16', 'rest_u_50', 'rest_u_84',
                  'rest_g_16', 'rest_g_50', 'rest_g_84',
                  'rest_i_16', 'rest_i_50', 'rest_i_84']

dict_thetas['rest_U'] = ['rest_U_16', 'rest_U_50', 'rest_U_84']
dict_thetas['rest_V'] = ['rest_V_16', 'rest_V_50', 'rest_V_84',]
dict_thetas['rest_J'] = ['rest_J_16', 'rest_J_50', 'rest_J_84',]
dict_thetas['rest_u'] = ['rest_u_16', 'rest_u_50', 'rest_u_84']
dict_thetas['rest_g'] = ['rest_g_16', 'rest_g_50', 'rest_g_84']
dict_thetas['rest_i'] = ['rest_i_16', 'rest_i_50', 'rest_i_84']


for i_t, t in enumerate(thetas):
    for i_dict, ii in enumerate(np.array([ii16, ii50, ii84])):
        _data, _mask = fill_col(dperc['rest_UVJugi'][:,i_t,ii])
        col_a = MaskedColumn(name=dict_thetas[t][i_dict], data=_data, mask=_mask, unit=u.ABmag)
        cat.add_columns([col_a])
    

In [ ]:
thetas = ['UV', 'VJ', 'gi', 'ug']
theta_colnames = ['UV_16', 'UV_50', 'UV_84',
                  'VJ_16', 'VJ_50', 'VJ_84',
                  'gi_16', 'gi_50', 'gi_84',
                  'ug_16', 'ug_50', 'ug_84']

dict_thetas['UV'] = ['UV_16', 'UV_50', 'UV_84']
dict_thetas['VJ'] = ['VJ_16', 'VJ_50', 'VJ_84']
dict_thetas['gi'] = ['gi_16', 'gi_50', 'gi_84']
dict_thetas['ug'] = ['ug_16', 'ug_50', 'ug_84']


for i_t, t in enumerate(thetas):
    for i_dict, ii in enumerate(np.array([ii16, ii50, ii84])):
        _data, _mask = fill_col(dperc['rest_UVJugi_colors'][:,i_t,ii])
        col_a = MaskedColumn(name=dict_thetas[t][i_dict], data=_data, mask=_mask, unit=u.ABmag)
        cat.add_columns([col_a])

In [ ]:
dperc_gz['rest_gz_2'].shape

In [ ]:
thetas = ['rest_g_sdss', 'rest_z_sdss']

theta_colnames = ['rest_g_sdss_16', 'rest_g_sdss_50', 'rest_g_sdss_84',
                  'rest_z_sdss_16', 'rest_sdss_50', 'rest_z_sdss_84']

dict_thetas['rest_g_sdss'] = ['rest_g_sdss_16', 'rest_g_sdss_50', 'rest_g_sdss_84']
dict_thetas['rest_z_sdss'] = ['rest_z_sdss_16', 'rest_z_sdss_50', 'rest_z_sdss_84']


for i_t, t in enumerate(thetas):
    for i_dict, ii in enumerate(np.array([ii16, ii50, ii84])):
        _data, _mask = fill_col(dperc_gz['rest_gz'][:,i_t,ii])
        col_a = MaskedColumn(name=dict_thetas[t][i_dict], data=_data, mask=_mask, unit=u.ABmag)
        cat.add_columns([col_a])
    

In [ ]:
thetas = ['gz_sdss']
theta_colnames = ['gz_sdss_16', 'gz_sdss_50', 'gz_sdss_84']

dict_thetas['gz_sdss'] = ['gz_sdss_16', 'gz_sdss_50', 'gz_sdss_84']

i_dict = 3
t = 'gz_sdss'
for i_dict, ii in enumerate(np.array([ii16, ii50, ii84])):
    _data, _mask = fill_col(dperc_gz['rest_gz_colors'][:,ii])
    col_a = MaskedColumn(name=dict_thetas[t][i_dict], data=_data, mask=_mask, unit=u.ABmag)
    cat.add_columns([col_a])

In [ ]:
_data, _mask = fill_col(dspec['chi2_fsps'], idx_finished=idx_finished_spec, idx_not_finished=idx_not_finished_spec)
col_a = MaskedColumn(name='chi2', data=_data, mask=_mask)
cat.add_columns([col_a])

In [ ]:
_data, _mask = fill_col(dspec['nbands'], idx_finished=idx_finished_spec, idx_not_finished=idx_not_finished_spec)
col_a = MaskedColumn(name='nbands', data=_data, mask=_mask)
cat.add_columns([col_a])

In [ ]:
cat[0]

# lensing

In [ ]:
idx_finished_lens = []
for idi in dlens['ID']:
    idx_finished_lens.append(int(idi)-1)
idx_finished_lens = np.array(idx_finished_lens)

# in first but not in second
id_not_finished_lens = list(set(cat['id']) - set(dlens['ID']))
idx_not_finished_lens = []
for idi in id_not_finished_lens:
    idx_not_finished_lens.append(int(idi)-1)
idx_not_finished_lens = np.array(idx_not_finished_lens)


def fill_col_lens(data, idx_finished=idx_finished_lens, idx_not_finished=idx_not_finished_lens):
    new_arr = np.ones(len(cat['id']))
    if len(idx_not_finished) > 0:
        new_arr[idx_not_finished] = np.nan
    
    new_arr[idx_finished] = np.copy(data)
    mask = np.ones_like(new_arr, dtype=bool)
    mask[idx_finished] = 0
    return new_arr, mask


lens_use_keys = ['mu', 'mu_low_68', 'mu_high_68', 
                 'mu_rad', 'mu_rad_low_68', 'mu_rad_high_68', 'mu_t', 'mu_t_low_68', 'mu_t_high_68', 
                 'gamma1', 'gamma1_low_68', 'gamma1_high_68', 'gamma2', 'gamma2_low_68', 'gamma2_high_68'
                ]

lens_use_colnames = ['mu', 'mu_68', 'mu_84', 
                     'mu_rad', 'mu_rad_68', 'mu_rad_84', 'mu_t', 'mu_t_68', 'mu_t_84', 
                     'gamma1', 'gamma1_68', 'gamma1_84', 'gamma2', 'gamma2_68', 'gamma2_84'
                ]

for i in range(len(lens_use_keys)):
    _data, _mask = fill_col_lens(dlens[lens_use_keys[i]])
    col_a = MaskedColumn(data=_data, name=lens_use_colnames[i], mask=_mask)
    cat.add_columns([col_a])

cat[cat['z_spec']>0.1]

In [ ]:
if 'fixed' in prior:
    fcat = '../sps_catalog/zspec_UNCOVER_{}_SPScatalog_{}.fits'.format(ver, sps_ver)
else:
    fcat = '../sps_catalog/UNCOVER_{}_SPScatalog_{}.fits'.format(ver, sps_ver)

cat.write(fcat, format='fits', overwrite=True)

In [ ]:
fcat

In [ ]:
# cat = Table.read(fcat)
# cat

# MAP spec

In [ ]:
import prospect.io.read_results as reader

In [ ]:
# read in an example
fname = 'chains/id_2475_mcmc_phisfh.h5'
fname = 'chains_parrot_alma_msa_v3.0.0_LW_SUPER_spsv1.0/id_36069_mcmc_phisfh.h5'
res, obs, _ = reader.results_from(fname, dangerous=False)
weff = obs['wave_effective']

In [ ]:
cat = Table.read('../phot_catalog/UNCOVER_{}_CATALOG.fits'.format(ver))
cat[0]

In [ ]:
# fperc = np.load('results/quant_phisfh_post_parrot_{}.npz'.format(fver), allow_pickle=True)
# fspec = np.load('results/spec_phisfh_post_parrot_{}.npz'.format(fver), allow_pickle=True)

# fperc = np.load('results/quant_phisfhfixzred_post_parrot_zfixed_v2.2.0_LW_spsv1.0.npz', allow_pickle=True)
# fspec = np.load('results/spec_phisfhfixzred_post_parrot_zfixed_v2.2.0_LW_spsv1.0.npz', allow_pickle=True)
# fspec.files

argsorted = np.argsort(fperc['objid'])
id_sort = fperc['objid'][argsorted]
dperc = {}
for key in fperc.files:
    dperc[key] = fperc[key][argsorted]

argsorted = np.argsort(fspec['objid'])
id_sort = fspec['objid'][argsorted]
dspec = {}
fspec.files.pop(-3)
for key in fspec.files:
    # if key not in ['eline_flux_map', 'eline_flux_med', 'chi2_parrot', 'parrot_mags']:
    if key not in ['chi2_parrot', 'parrot_mags']:
        dspec[key] = fspec[key][argsorted]

In [ ]:
c_ind, a_ind, b_ind = np.intersect1d(dperc['objid'], dspec['objid'], assume_unique=True, return_indices=True)

In [ ]:
from prospect.sources import FastStepBasis
from prospect.models.sedmodel import PolySpecModel

def load_sps(zcontinuous=2, compute_vega_mags=False, **extras):
    sps = FastStepBasis(zcontinuous=zcontinuous,
                        compute_vega_mags=compute_vega_mags)  # special to remove redshifting issue
    return sps

sps = load_sps()

In [ ]:
if 'fixed' in prior:
    fnpz = '../sps_catalog/ancillaries/seds_map_zspec_{}_{}.npz'.format(ver, sps_ver)
else:
    fnpz = '../sps_catalog/ancillaries/seds_map_{}_{}.npz'.format(ver, sps_ver)

print(fnpz)

np.savez(fnpz, objid=dspec['objid'][b_ind], zred=dperc['zred_ml'][a_ind],
         mu=dspec['mu_map'][b_ind], obsmags=dspec['obsmag'][b_ind], obsunc=dspec['obsmag_unc'][b_ind], 
         modmags=dspec['modmag_map'][b_ind], modspec=dspec['modspec_map'][b_ind], 
         wavspec=sps.wavelengths, weff=weff)
